# Notebook 3: Evaluate & Deploy

**Phase 5** | ~20 minutes

In this notebook you will:
1. Evaluate your trained model — detection metrics (mAP50) AND business metrics (compliance accuracy)
2. Run the compliance postprocessor — translate detections into actionable safety insights
3. Benchmark inference speed — can your model run at 30+ FPS?
4. Compare approaches and discuss what you'd change

> **Key insight**: Model metrics (mAP50) ≠ business metrics (compliance accuracy).
> The construction site manager doesn't care about mAP — they care about "is everyone safe?"

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"

# Set this to YOUR trained model
MODEL_PATH = None  # e.g., DATA_DIR / "ppe_results" / "my_experiment" / "weights" / "best.pt"

# Set this to your labeled dataset
EVAL_DATASET = None  # 2-class dataset (for mAP)
GT_DATASET = None    # 3-class dataset (for compliance GT, if available)

print("Set MODEL_PATH, EVAL_DATASET, and GT_DATASET above before continuing.")
print("Or ask Claude Code: 'Help me set up the evaluation paths'")

---
## 1. Detection Evaluation (mAP50)

Standard object detection metrics:
- **mAP50**: Mean Average Precision at IoU ≥ 0.5
- **Per-class AP**: How well do we detect each class?
- **Precision/Recall**: The usual tradeoff

### Tool Reference

```bash
python data/evaluate_2class_experiments.py \
    --model path/to/best.pt \
    --eval-dataset path/to/2class_dataset \
    --gt-dataset path/to/3class_gt_dataset \
    --mode exp_a \
    --conf 0.25
```

In [ ]:
# TODO: Run evaluation
# Ask Claude Code: "Evaluate my trained model and show detection + compliance metrics"


---
## 2. Compliance Post-Processing

Detection tells us WHERE objects are. Compliance tells us WHETHER workers are safe.

### How it works

For each detected `person`:
1. Define the **head region** = top 40% of the person bounding box
2. Check if ANY detected `hardhat` overlaps the head region with IoU ≥ 0.1
3. If yes → **compliant** (wearing a hard hat)
4. If no → **non-compliant** (missing hard hat)

### Tool Reference

```bash
# Single image
python data/compliance_postprocessor.py \
    --model path/to/best.pt \
    --image path/to/image.jpg \
    --verbose

# Batch processing
python data/compliance_postprocessor.py \
    --model path/to/best.pt \
    --source-dir data/synthetic_ppe \
    --conf 0.25
```

In [ ]:
# ── Utility: Visualize compliance results ────────────────────────────────────

def visualize_compliance(image_path, model_path, conf=0.25, head_fraction=0.4,
                         min_overlap=0.1, figsize=(14, 10)):
    """Run compliance check and visualize results on a single image.
    
    Green box = compliant (wearing hard hat)
    Red box = non-compliant (no hard hat detected)
    Small green box = detected hard hat
    Yellow dashed = head region
    """
    from ultralytics import YOLO
    
    model = YOLO(str(model_path))
    results = model(str(image_path), conf=conf, verbose=False)
    
    img = np.array(Image.open(image_path))
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        ax.set_title("No detections", fontsize=12)
        ax.axis("off")
        plt.show()
        return
    
    # Separate persons and hardhats
    persons = []
    hardhats = []
    for box in boxes:
        cls_id = int(box.cls[0])
        xyxy = box.xyxy[0].cpu().numpy()
        conf_val = float(box.conf[0])
        if cls_id == 1:  # person
            persons.append((xyxy, conf_val))
        elif cls_id == 0:  # hardhat
            hardhats.append((xyxy, conf_val))
    
    # Draw hardhats (small green boxes)
    for (x1, y1, x2, y2), c in hardhats:
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                             linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-4, f"hardhat {c:.2f}", color="lime", fontsize=7,
                bbox=dict(facecolor="black", alpha=0.7, pad=1))
    
    # Check compliance for each person
    compliant_count = 0
    total_persons = len(persons)
    
    for (px1, py1, px2, py2), pc in persons:
        # Head region = top fraction of person bbox
        head_y2 = py1 + (py2 - py1) * head_fraction
        
        # Check overlap with any hardhat
        is_compliant = False
        for (hx1, hy1, hx2, hy2), hc in hardhats:
            # IoU between hardhat and head region
            ix1 = max(hx1, px1)
            iy1 = max(hy1, py1)
            ix2 = min(hx2, px2)
            iy2 = min(hy2, head_y2)
            
            if ix1 < ix2 and iy1 < iy2:
                intersection = (ix2 - ix1) * (iy2 - iy1)
                hat_area = (hx2 - hx1) * (hy2 - hy1)
                head_area = (px2 - px1) * (head_y2 - py1)
                union = hat_area + head_area - intersection
                iou = intersection / union if union > 0 else 0
                if iou >= min_overlap:
                    is_compliant = True
                    break
        
        # Draw person box
        color = "green" if is_compliant else "red"
        status = "SAFE" if is_compliant else "UNSAFE"
        rect = plt.Rectangle((px1, py1), px2-px1, py2-py1,
                             linewidth=3, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(px1, py2+12, f"{status} ({pc:.2f})", color=color, fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.7, pad=2))
        
        # Draw head region (dashed yellow)
        head_rect = plt.Rectangle((px1, py1), px2-px1, head_y2-py1,
                                  linewidth=1, edgecolor="yellow", facecolor="none",
                                  linestyle="--")
        ax.add_patch(head_rect)
        
        if is_compliant:
            compliant_count += 1
    
    # Summary
    rate = compliant_count / total_persons * 100 if total_persons > 0 else 0
    ax.set_title(
        f"Compliance: {compliant_count}/{total_persons} workers safe ({rate:.0f}%) | "
        f"{len(hardhats)} hardhats detected",
        fontsize=12, fontweight="bold"
    )
    ax.axis("off")
    
    # Legend
    legend_elements = [
        mpatches.Patch(edgecolor="green", facecolor="none", linewidth=2, label="Compliant"),
        mpatches.Patch(edgecolor="red", facecolor="none", linewidth=2, label="Non-compliant"),
        mpatches.Patch(edgecolor="lime", facecolor="none", linewidth=2, label="Hard hat"),
        mpatches.Patch(edgecolor="yellow", facecolor="none", linewidth=1,
                       linestyle="--", label="Head region"),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO: Run compliance check on some images
# Example:
#   images = sorted(Path(DATA_DIR / "synthetic_ppe").rglob("*.jpg"))[:5]
#   for img_path in images:
#       visualize_compliance(img_path, MODEL_PATH)
#
# Ask Claude Code: "Run the compliance postprocessor on my validation images"


---
## 3. Inference Speed Benchmark

Production requirement: **30+ FPS** (< 33ms per image).

```bash
python data/benchmark_inference_speed.py \
    --device cuda \
    --max-images 10
```

This compares **YOLOe-26n** (foundation model) vs **SAM3** to show the speed gap
between foundation models and distilled student models.

In [ ]:
# TODO: Benchmark inference speed
# Ask Claude Code: "Benchmark the inference speed of my model vs the foundation model"


---
## 4. Model Comparison

Let's compare all the approaches we've tried.

In [ ]:
# ── Utility: Build comparison table ──────────────────────────────────────────

import pandas as pd

def build_comparison_table(results_list):
    """Build a comparison table from a list of result dictionaries.
    
    Each dict should have: name, map50, map50_95, speed_ms, compliance_accuracy
    Missing values are OK — they'll show as '—'.
    """
    rows = []
    for r in results_list:
        rows.append({
            "Model": r.get("name", "Unknown"),
            "mAP50": f"{r['map50']:.3f}" if r.get("map50") else "—",
            "mAP50-95": f"{r['map50_95']:.3f}" if r.get("map50_95") else "—",
            "Speed (ms)": f"{r['speed_ms']:.1f}" if r.get("speed_ms") else "—",
            "FPS": f"{1000/r['speed_ms']:.0f}" if r.get("speed_ms") else "—",
            "Compliance Acc.": f"{r['compliance_accuracy']:.1%}" if r.get("compliance_accuracy") else "—",
        })
    
    df = pd.DataFrame(rows)
    return df

# Example usage (fill in your actual numbers):
# results = [
#     {"name": "YOLOe-26n (zero-shot)", "map50": 0.45, "speed_ms": 85.0},
#     {"name": "YOLO26n (SAM3 labels)", "map50": 0.63, "speed_ms": 4.2, "compliance_accuracy": 0.82},
#     {"name": "YOLO26n (GDINO labels)", "map50": 0.54, "speed_ms": 4.2, "compliance_accuracy": 0.75},
# ]
# build_comparison_table(results)

In [ ]:
# TODO: Build your comparison table with actual results
# Ask Claude Code: "Build a comparison table of all the models we've evaluated"


---
## 5. Discussion & Takeaways

### Discussion Prompts

1. **Prompt engineering**: What if you changed the auto-labeling prompts? Would "helmet" vs "hard hat" make a difference? (Hint: it does — 2.2x!)

2. **3-class vs 2-class**: Why is detecting `no_hardhat` (an absence) fundamentally harder than detecting `hardhat` + `person` and deriving compliance?

3. **More data vs better labels**: You saw the saturating curve — adding more images with noisy labels barely helps. What's the right investment: more images or better labeling?

4. **Business metrics**: mAP50 measures detection accuracy. Compliance accuracy measures safety. When might a model with lower mAP50 have better compliance accuracy?

5. **Edge cases**: Which scene categories were hardest? What would you do about CCTV (far away, small workers) vs close-up scenes?

### Key Takeaways

| Lesson | Evidence |
|--------|----------|
| Label quality > data quantity | Saturating curve: mAP50 asymptote at 0.527 with noisy labels, broken to 0.633 with curated labels |
| Prompt engineering is first-order | `"helmet. person."` finds 2.2x more helmets than `"hard hat. safety helmet. person."` |
| Business metrics ≠ model metrics | mAP50 can be high while compliance accuracy is low (or vice versa) |
| Foundation models → distillation | SAM3/YOLOE label data → train YOLO26n for 30+ FPS production deployment |
| 2-class + post-processing > 3-class | Detecting absences is fundamentally harder than detecting objects |
| Inspect before training | 35.6% of auto-labels were noise (tiny labels) |

### What would you change?

Write your ideas:
- [ ] ...
- [ ] ...
- [ ] ...